# Notebook 3: ElasticNet
Objetivo: `km_media_prox28`. 6 corridas de ElasticNet: mini-rejilla de 3 valores de alpha x 2 valores de `l1_ratio`.
El baseline NO se vuelve a registrar: ya existe en el Notebook 1. Aquí solo se recalculan sus métricas para comparar.

In [1]:
import sys
sys.path.append("../src")
import itertools
import pandas as pd
from sklearn.linear_model import ElasticNet
import modelos_comunes as mc

mc.configurar_mlflow()

MLflow: http://34.229.81.56:5000 | experimento: modelo_km_dia


## 1. Datos

In [2]:
train, test = mc.cargar_datos()
print("train:", train.shape, "| test:", test.shape)
print("version datos:", mc.version_datos())

train: (462, 28) | test: (68, 28)
version datos: ce34fbfc258e270aeff248a8ed7435ed


## 2. Métricas del baseline (solo cálculo, sin registrar otra corrida)

In [3]:
base = mc.calcular_metricas(test[mc.OBJETIVO].values, test["km_media28"].values)
print(f"Baseline -> MAE {base['mae']:.2f} km/dia | MAE fecha {base['mae_fecha_dias']:.2f} dias")

Baseline -> MAE 15.30 km/dia | MAE fecha 1.92 dias


## 3. ElasticNet: alpha x l1_ratio
`l1_ratio` cerca de 0 se parece a Ridge; cerca de 1 se parece a Lasso. En Ridge los mejores resultados estuvieron en alpha pequeño, por eso la rejilla se concentra en valores bajos.

In [4]:
alphas = [0.001, 0.01, 0.1]
l1_ratios = [0.2, 0.8]
resultados = []
for a, l1 in itertools.product(alphas, l1_ratios):
    r = mc.correr_modelo("elasticnet", f"elasticnet_alpha_{a}_l1_{l1}",
                         ElasticNet(alpha=a, l1_ratio=l1, max_iter=20000),
                         {"alpha": a, "l1_ratio": l1}, train, test, base)
    resultados.append(r)

2026/11/07 18:16:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.001_l1_0.2 at: http://34.229.81.56:5000/#/experiments/1/runs/8d95b4de57b243aca51ff60efd7051b9
🧪 View experiment at: http://34.229.81.56:5000/#/experiments/1
elasticnet_alpha_0.001_l1_0.2 MAE 12.56 | CV 19.46 | mejora 17.9% | fecha 2.88 d


2026/11/07 18:16:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.001_l1_0.8 at: http://34.229.81.56:5000/#/experiments/1/runs/367a8ed82d004dc09854fddccc3cc6e9
🧪 View experiment at: http://34.229.81.56:5000/#/experiments/1
elasticnet_alpha_0.001_l1_0.8 MAE 12.45 | CV 19.41 | mejora 18.6% | fecha 2.87 d


2026/11/07 18:16:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.01_l1_0.2 at: http://34.229.81.56:5000/#/experiments/1/runs/49d30035dd9a42ca958a02cf6b5a9d37
🧪 View experiment at: http://34.229.81.56:5000/#/experiments/1
elasticnet_alpha_0.01_l1_0.2 MAE 13.70 | CV 21.50 | mejora 10.4% | fecha 2.98 d


... (2 outputs left out)

2026/11/07 18:16:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.1_l1_0.2 at: http://34.229.81.56:5000/#/experiments/1/runs/a4e1027621ad4cee8aeed63cb2f53362
🧪 View experiment at: http://34.229.81.56:5000/#/experiments/1
elasticnet_alpha_0.1_l1_0.2  MAE 15.40 | CV 25.54 | mejora -0.7% | fecha 3.26 d


2026/11/07 18:16:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run elasticnet_alpha_0.1_l1_0.8 at: http://34.229.81.56:5000/#/experiments/1/runs/608b1575c8bc48f09e4ed9712cefa863
🧪 View experiment at: http://34.229.81.56:5000/#/experiments/1
elasticnet_alpha_0.1_l1_0.8  MAE 14.61 | CV 23.27 | mejora 4.5% | fecha 3.06 d


## 4. Resumen

In [5]:
tabla = pd.DataFrame(resultados)[["run_name", "alpha", "l1_ratio", "cv_mae", "mae", "rmse",
                                  "mejora_mae_pct", "mae_fecha_dias", "mejora_fecha_pct"]]
tabla.sort_values("cv_mae").round(3)

,run_name,alpha,l1_ratio,cv_mae,mae,rmse,mejora_mae_pct,mae_fecha_dias,mejora_fecha_pct
1,elasticnet_alpha_0.001_l1_0.8,0.001,0.8,19.411,12.449,13.824,18.610,2.869,-49.269
0,elasticnet_alpha_0.001_l1_0.2,0.001,0.2,19.461,12.564,13.957,17.860,2.881,-49.877
3,elasticnet_alpha_0.01_l1_0.8,0.010,0.8,19.957,12.822,14.249,16.172,2.902,-50.940
2,elasticnet_alpha_0.01_l1_0.2,0.010,0.2,21.502,13.702,15.384,10.418,2.984,-55.206
5,elasticnet_alpha_0.1_l1_0.8,0.100,0.8,23.271,14.612,16.702,4.469,3.063,-59.349
4,elasticnet_alpha_0.1_l1_0.2,0.100,0.2,25.536,15.405,17.825,-0.712,3.257,-69.434
